# Digital Cash Management Forecast: daily flows viewer

Reads the Parquet files of `daily_flows` (what staging writes and every model gets: one row per client, day and
category with `money_in`, `money_out` and `postings`) and shows them. It only reads: nothing here changes the
project. In the load test project (import it there), set `FOLDER` to `load_test_flows` to look at its fake data.

**Kernel:** the project's code env (the setup's `CODE_ENV`, which has polars), like `forecast_lab.ipynb`.

In [ ]:
FOLDER = "daily_flows"       # the folder to read: daily_flows (the forecast project) or load_test_flows (the load test project)
SIZE = ""                    # load_test_flows only: the size of fake data to read, its subfolder name (empty: the first)
SUMMARY_GROUPS = 5           # how many groups of clients the summary reads (-1: all of them; slower on the full data)
SAMPLE_ROWS = 20             # rows shown from the first file
CLIENT_NO = ""               # a client number to show day by day (empty: the first client of the first file)

In [ ]:
# Reads only: nothing in this notebook changes the project.
import time

import pandas as pd
import polars as pl

from cashflow import client_groups, read_group, read_manifest, read_parquet

folder, manifest = read_manifest(FOLDER)
if manifest is None:          # the load test: one subfolder per size of fake data
    sizes = sorted({p.split("/")[1] for p in folder.list_paths_in_partition() if p.endswith("/manifest.json")
                    and p.count("/") == 2})
    if sizes:
        print("Sizes of fake data here:", ", ".join(sizes))
        size = SIZE or sizes[0]
        folder, manifest = read_manifest(FOLDER, f"/{size}")
        FOLDER = f"{FOLDER}/{size}"
if manifest is None:
    raise RuntimeError(f"Nothing in the folder {FOLDER}: run staging (or, in the load test project, the scenario Run "
                       "the load test) first")
files = pd.DataFrame([(int(g), month, path, rows) for g, months in manifest["files"].items()
                      for month, (path, rows) in months.items()],
                     columns=["group", "month", "file", "rows"]).sort_values(["group", "month"]).reset_index(drop=True)
print(f"{FOLDER}: {manifest['rows']:,} rows in {len(files):,} files: {manifest['groups']:,} groups of clients (a client "
      f"never changes group), one file per group and month, {files['month'].min()} to {files['month'].max()}")

## 1. Files

One Parquet file per group of clients and month: rows per month, and the first files.

In [ ]:
# Rows per month (all the groups together), from the manifest: no file is read here.
display(files.groupby("month").agg(files=("file", "size"), rows=("rows", "sum")))
display(files.head(SAMPLE_ROWS))

## 2. Summary

Rows, clients, postings and money per category, over the first `SUMMARY_GROUPS` groups of clients.

In [ ]:
# Totals over the first SUMMARY_GROUPS groups of clients (each client's rows are all in its group), one group at a time.
groups = sorted(int(g) for g in manifest["files"])
chosen = groups if SUMMARY_GROUPS in (-1, None) else groups[:SUMMARY_GROUPS]
started, parts = time.time(), []
for i, group in enumerate(chosen, 1):
    parts.append(read_group(folder, manifest, group).group_by("category").agg(
        rows=pl.len(), clients=pl.col("client_no").n_unique(), postings=pl.col("postings").sum(),
        money_in=pl.col("money_in").sum(), money_out=pl.col("money_out").sum(),
        first_day=pl.col("day").min(), last_day=pl.col("day").max()))
    if i % 10 == 0 or i == len(chosen):
        print(f"read {i} of {len(chosen)} groups ({time.time() - started:.0f} s)")
by_category = (pl.concat(parts).group_by("category").agg(
    pl.col("rows", "clients", "postings", "money_in", "money_out").sum(), pl.col("first_day").min(),
    pl.col("last_day").max()).with_columns(net=pl.col("money_in") - pl.col("money_out")).sort("rows", descending=True))
print(f"{'All' if len(chosen) == len(groups) else f'The first {len(chosen)} of {len(groups)}'} groups: "
      f"{by_category['rows'].sum():,} rows, {by_category['clients'].sum():,} client-categories")
display(by_category.to_pandas().round(0))

## 3. Sample rows

The columns and the first rows of the first file.

In [ ]:
first = read_parquet(folder, files["file"].iloc[0]).to_pandas()
print(f"{files['file'].iloc[0]}: {len(first):,} rows, {first['client_no'].nunique():,} clients. Columns:")
print(first.dtypes.to_string())
display(first.head(SAMPLE_ROWS))

## 4. One client

`CLIENT_NO` day by day, and per month.

In [ ]:
# One client's daily rows and its net cash flow per month (money in, money out, net): only its group's files are read.
wanted = str(CLIENT_NO).strip() or first["client_no"].iloc[0]
group = client_groups([wanted], manifest["groups"])[0]
rows = read_group(folder, manifest, group).filter(pl.col("client_no") == wanted).sort("day", "category").to_pandas()
if rows.empty:
    print(f"Client {wanted} is not in {FOLDER}")
else:
    print(f"Client {wanted} (group {group}): {len(rows):,} rows, {rows['day'].min():%Y-%m-%d} to "
          f"{rows['day'].max():%Y-%m-%d}, {rows['postings'].sum():,} postings")
    display(rows.tail(SAMPLE_ROWS))
    monthly = (rows.assign(month=rows["day"].dt.to_period("M").dt.to_timestamp())
               .groupby("month")[["money_in", "money_out"]].sum())
    monthly["net"] = monthly["money_in"] - monthly["money_out"]
    display(monthly.round(0))
    ax = monthly.plot(figsize=(12, 4), title=f"Client {wanted}: money in, money out and net per month (SAR)")
    ax.axhline(0, color="grey", linewidth=0.8)